# Sprocket's editor-token model

This notebook trains a small model to turn requests ("when the player presses jump, play the jump sound") plus a
snapshot of the editor into **editor tokens**: Schematics commands where every node type and Behavior is a single
token. Sprocket's server turns those tokens back into editor commands and a sentence to say.

**What you need**
1. In Unity: **Tools > Remedy > Schematics > Assistant > Training > Generate Training Data…**. It writes `vocab.tsv`
   and `dataset.jsonl` to `Agent~/Training/data/`.
2. A GPU runtime: in Colab, *Runtime > Change runtime type > T4 GPU* (free tier works); on Kaggle, turn on a GPU
   accelerator and add the two files as a dataset.
3. Run the cells top to bottom (*Runtime > Run all*). At the end you download two files:
   `sprocket-macro.gguf` and `sprocket-macro.vocab.tsv`. Put both in Sprocket's `models` folder (next to the other
   models; Preferences > Schematics Assistant shows the install folder) and restart the server. The log says
   `[macro] Loaded the editor-token model`.

**How it works**
- The model starts as a small pretrained model (it already reads English).
- Every token in `vocab.tsv` is added to its vocabulary. Each new token starts out meaning roughly "the words that describe
  it" (its embedding is the average of those words'), so `<…/node/OnUpdateNode>` already leans toward "on update".
- It's fine-tuned on the examples: the prompt is the editor state + request, and it learns to write only the answer.
- Optionally, a bigger model first rewords the requests a few ways, so it understands how people actually talk.
- It's checked on examples it never saw (exact match of the whole answer), then saved as GGUF (what Sprocket runs).

In [ ]:
#@title Settings
BASE_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"  #@param ["HuggingFaceTB/SmolLM2-360M-Instruct", "HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"]
EPOCHS = 2            #@param {type:"number"}
LEARNING_RATE = 5e-5  #@param {type:"number"}
BATCH_SIZE = 8        #@param {type:"integer"}
MAX_LENGTH = 1024     #@param {type:"integer"}
# Reword some requests with a bigger model first (slower, but understands real phrasing much better).
PARAPHRASE = True     #@param {type:"boolean"}
PARAPHRASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  #@param {type:"string"}
PARAPHRASE_FRACTION = 0.3  #@param {type:"number"}
PARAPHRASES_EACH = 2  #@param {type:"integer"}
# GGUF compression: q8_0 (small, near-lossless) or f16.
GGUF_TYPE = "q8_0"    #@param ["q8_0", "f16"]

In [ ]:
!pip -q install "transformers>=4.45" "datasets>=3.0" "accelerate>=1.0" sentencepiece
import os, sys, json, random, math, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (turn on a GPU runtime!)")

In [ ]:
#@title Get the data (upload in Colab, or a Kaggle dataset)
DATA = "/content/data" if os.path.exists("/content") else "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
if os.path.exists("/kaggle/input"):
    # Kaggle: find the files in any attached dataset.
    for root, _, files in os.walk("/kaggle/input"):
        for name in files:
            if name in ("vocab.tsv", "dataset.jsonl"):
                os.system(f'cp "{os.path.join(root, name)}" "{DATA}/"')
else:
    try:
        from google.colab import files
        if not all(os.path.exists(f"{DATA}/{n}") for n in ("vocab.tsv", "dataset.jsonl")):
            print("Upload vocab.tsv and dataset.jsonl (from Agent~/Training/data):")
            for name, content in files.upload().items():
                open(f"{DATA}/{name}", "wb").write(content)
    except ImportError:
        print(f"Put vocab.tsv and dataset.jsonl in {DATA}")

vocab = []
for line in open(f"{DATA}/vocab.tsv", encoding="utf-8"):
    if line.startswith("#") or not line.strip():
        continue
    token, kind, assembly, type_name, name, description = (line.rstrip("\n").split("\t") + [""] * 6)[:6]
    vocab.append(dict(token=token, kind=kind, assembly=assembly, type=type_name, name=name, description=description))
examples = [json.loads(line) for line in open(f"{DATA}/dataset.jsonl", encoding="utf-8") if line.strip()]
print(f"{len(vocab)} tokens, {len(examples)} examples")
print(examples[0]["prompt"]); print(examples[0]["completion"])

In [ ]:
#@title Reword requests with a bigger model (optional)
import re
def request_of(prompt):
    m = re.search(r"<request> (.*)\n<macro>$", prompt)
    return m.group(1) if m else None

if PARAPHRASE:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    ptok = AutoTokenizer.from_pretrained(PARAPHRASE_MODEL)
    ptok.padding_side = "left"
    pmodel = AutoModelForCausalLM.from_pretrained(PARAPHRASE_MODEL, torch_dtype=torch.float16, device_map="cuda")
    chosen = random.Random(7).sample(examples, int(len(examples) * PARAPHRASE_FRACTION))
    added = []
    for start in range(0, len(chosen), 32):
        batch = chosen[start:start + 32]
        prompts = []
        for ex in batch:
            req = request_of(ex["prompt"])
            messages = [
                {"role": "system", "content": "You reword requests to a game-editor assistant. Keep every name, number and quoted text exactly. Reply with one reworded request per line, nothing else."},
                {"role": "user", "content": f"Reword this {PARAPHRASES_EACH} different casual ways:\n{req}"},
            ]
            prompts.append(ptok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        inputs = ptok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = pmodel.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.8, top_p=0.9)
        for ex, seq in zip(batch, out):
            text = ptok.decode(seq[inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for line in [l.strip(" -*\t0123456789.\"") for l in text.splitlines()][:PARAPHRASES_EACH]:
                if 3 <= len(line) <= 200 and "<" not in line:
                    added.append(dict(ex, prompt=ex["prompt"].replace(f"<request> {request_of(ex['prompt'])}\n", f"<request> {line}\n"), kind=ex["kind"] + "+reworded"))
        print(f"{start + len(batch)}/{len(chosen)} reworded", end="\r")
    examples += added
    print(f"\nAdded {len(added)} reworded examples; {len(examples)} in all")
    del pmodel; torch.cuda.empty_cache()

In [ ]:
#@title Load the base model and add the editor tokens
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32).cuda()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

new_tokens = [v["token"] for v in vocab if v["token"] not in tokenizer.get_vocab()]
old_size = len(tokenizer)
tokenizer.add_tokens(new_tokens, special_tokens=False)
model.resize_token_embeddings(len(tokenizer))

# Each new token starts as the average of the words that describe it.
embeddings = model.get_input_embeddings().weight
head = model.get_output_embeddings().weight if model.get_output_embeddings() is not None else None
tied = head is not None and head.data_ptr() == embeddings.data_ptr()
by_token = {v["token"]: v for v in vocab}
with torch.no_grad():
    mean = embeddings[:old_size].mean(0)
    for token in new_tokens:
        entry = by_token[token]
        words = (entry["description"] or entry["name"] or token.strip("<>")).strip()
        ids = tokenizer(words, add_special_tokens=False)["input_ids"]
        ids = [i for i in ids if i < old_size]
        tid = tokenizer.convert_tokens_to_ids(token)
        embeddings[tid] = embeddings[ids].mean(0) if ids else mean
        if head is not None and not tied:
            head[tid] = head[ids].mean(0) if ids else head[:old_size].mean(0)
print(f"Added {len(new_tokens)} tokens (vocabulary {old_size} -> {len(tokenizer)}); tied embeddings: {tied}")
print("Check:", tokenizer.tokenize(" <connect> <ref:3> <flow>"))

In [ ]:
#@title Tokenize (the model only learns to write the answer)
random.Random(42).shuffle(examples)
split = max(50, len(examples) // 20)
eval_examples, train_examples = examples[:split], examples[split:]

def encode(ex):
    prompt_ids = tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(ex["completion"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    ids = prompt_ids + answer_ids
    if len(ids) > MAX_LENGTH:
        return None
    return {"input_ids": ids, "labels": [-100] * len(prompt_ids) + answer_ids}

train_set = [e for e in (encode(x) for x in train_examples) if e]
eval_set = [e for e in (encode(x) for x in eval_examples) if e]
lengths = sorted(len(e["input_ids"]) for e in train_set)
print(f"{len(train_set)} training, {len(eval_set)} held out; length median {lengths[len(lengths)//2]}, max {lengths[-1]}")

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]),
    }

In [ ]:
#@title Train
from transformers import Trainer, TrainingArguments
model.gradient_checkpointing_enable()
model.config.use_cache = False
args = TrainingArguments(
    output_dir="/tmp/sprocket-macro", num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE, gradient_accumulation_steps=2,
    warmup_ratio=0.03, lr_scheduler_type="cosine", weight_decay=0.0,
    fp16=True, logging_steps=50, eval_strategy="steps", eval_steps=500, save_strategy="no", report_to=[],
    group_by_length=True, dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=args, train_dataset=train_set, eval_dataset=eval_set, data_collator=collate)
trainer.train()

In [ ]:
#@title Check it on examples it never saw (whole answer must match)
model.eval(); model.config.use_cache = True
tokenizer.padding_side = "left"
def norm(text): return " ".join(text.replace("<end>", "").split())
held = eval_examples[:200]
right, shown = 0, 0
for start in range(0, len(held), 16):
    batch = held[start:start + 16]
    inputs = tokenizer([b["prompt"] for b in batch], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    for ex, seq in zip(batch, out):
        answer = tokenizer.decode(seq[inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        answer = answer.split("<end>")[0]
        ok = norm(answer) == norm(ex["completion"])
        right += ok
        if not ok and shown < 5:
            shown += 1
            print("REQUEST:", re.search(r"<request> (.*)\n", ex["prompt"]).group(1))
            print("  wanted:", norm(ex["completion"])); print("  wrote: ", norm(answer)); print()
print(f"Exact match: {right}/{len(held)} = {right / len(held):.0%}")

In [ ]:
#@title Save as GGUF and download
OUT = "/tmp/sprocket-macro-hf"
model.save_pretrained(OUT, safe_serialization=True)
tokenizer.save_pretrained(OUT)
if not os.path.exists("/tmp/llama.cpp"):
    os.system("git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    os.system("pip -q install -r /tmp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")
FINAL = "/content" if os.path.exists("/content") else "/kaggle/working"
code = os.system(f'python /tmp/llama.cpp/convert_hf_to_gguf.py {OUT} --outfile {FINAL}/sprocket-macro.gguf --outtype {GGUF_TYPE}')
assert code == 0, "GGUF conversion failed (see the output above)"
os.system(f'cp "{DATA}/vocab.tsv" "{FINAL}/sprocket-macro.vocab.tsv"')
print("Done:", os.path.getsize(f"{FINAL}/sprocket-macro.gguf") // (1024 * 1024), "MB")
try:
    from google.colab import files
    files.download(f"{FINAL}/sprocket-macro.gguf")
    files.download(f"{FINAL}/sprocket-macro.vocab.tsv")
except ImportError:
    print(f"Kaggle: download sprocket-macro.gguf and sprocket-macro.vocab.tsv from {FINAL} (Output panel).")